# Multigrid recon comparison

Both anatomies, both accelerations. `lpdsnet` is the flat (non-multigrid)
counterpart of `mglpds`; `varnet` is an outside baseline. The LADMM pair
(`altsplit` / `mgaltsplit`) is left out — add them back to `MODELS` below.

Reads the CSVs written by `scripts/evaluate.py`. Produce them first:

```
python scripts/evaluate.py --eval-config config/eval/knee.json
python scripts/evaluate.py --eval-config config/eval/brain.json
```

In [ ]:
import csv, json, math, os, pathlib, sys
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)

MODELS = ["lpdsnet", "mglpds", "mggrouplpds", "varnet"]
METRICS = [("psnr", "PSNR", "{:.2f}", True),      # (key, label, fmt, higher_is_better)
           ("ssim", "SSIM", "{:.4f}", True),
           ("nrmse", "NRMSE", "{:.4f}", False),
           ("lpips", "LPIPS", "{:.4f}", False)]


def is_multigrid(anatomy, model, R):
    '''True when the config's K is [K_outer, [i0, ...]] rather than a bare int.'''
    p = ROOT / "config" / anatomy / "mg" / f"{model}_R{R}.json"
    if not p.exists():
        return None
    params = json.load(open(p))["model"]["params"]
    K = params.get("K", params.get("denoiser_kws", {}).get("K"))
    return isinstance(K, list)


def _rows_from(path):
    if not pathlib.Path(path).exists():
        return []
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def load(anatomy, R):
    '''-> (rows, missing). Prefers the per-anatomy aggregate CSV, else per-model.'''
    rows, missing = [], []
    agg = {r.get("run", ""): r for r in _rows_from(f"results/eval/{anatomy}.csv")}
    for m in MODELS:
        key = f"{m}_R{R}"
        r = agg.get(key)
        if r is None:
            per = _rows_from(f"results/eval/{anatomy}/{key}.csv")
            r = per[0] if per else None
        if r is None:
            missing.append(key)
            continue
        r = dict(r)
        r["model"] = m
        r["mg"] = is_multigrid(anatomy, m, R)
        rows.append(r)
    return rows, missing


def _f(r, k):
    try:
        v = float(r.get(k, ""))
        return v if v == v else None
    except (TypeError, ValueError):
        return None


def show(anatomy, R, plot=True):
    rows, missing = load(anatomy, R)
    title = f"{anatomy}  R={R}"
    if not rows:
        print(f"{title}: no results yet.")
        if missing:
            print("  missing:", ", ".join(missing))
            print(f"  run: python scripts/evaluate.py --eval-config config/eval/{anatomy}.json")
        return rows

    rows.sort(key=lambda r: -(_f(r, "psnr_mean") or -math.inf))
    best = {k: (max if hi else min)(
                [v for v in (_f(r, f"{k}_mean") for r in rows) if v is not None],
                default=None)
            for k, _, _, hi in METRICS}

    head = f"{'model':<14}{'mg':>4}{'params':>10}"
    for _, lab, _, _ in METRICS:
        head += f"{lab:>12}"
    head += f"{'n':>6}"
    print(f"=== {title} ===")
    print(head)
    print("-" * len(head))
    for r in rows:
        mg = {True: "yes", False: "-", None: "?"}[r["mg"]]
        npar = _f(r, "n_params")
        line = f"{r['model']:<14}{mg:>4}{(f'{npar/1e6:.2f}M' if npar else '?'):>10}"
        for k, _, fmt, _hi in METRICS:
            v = _f(r, f"{k}_mean")
            cell = fmt.format(v) if v is not None else "--"
            if v is not None and best[k] is not None and abs(v - best[k]) < 1e-9:
                cell += "*"                      # best in column
            line += f"{cell:>12}"
        line += f"{r.get('n_slices', '?'):>6}"
        print(line)
    if missing:
        print("missing:", ", ".join(missing))

    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(11, 0.5 + 0.42 * len(rows)))
        names = [r["model"] for r in rows]
        cols = ["#d62728" if r["mg"] else "#7f7f7f" for r in rows]
        for a, (k, lab, _, hi) in zip(ax, [METRICS[0], METRICS[1]]):
            vals = [_f(r, f"{k}_mean") or 0.0 for r in rows]
            errs = [_f(r, f"{k}_std") or 0.0 for r in rows]
            a.barh(names, vals, xerr=errs, color=cols)
            a.invert_yaxis()
            a.set_xlabel(lab)
            lo = min(v for v in vals if v) if any(vals) else 0
            a.set_xlim(left=max(0, lo - 0.12 * (max(vals) - lo + 1e-9)))
            a.grid(axis="x", alpha=.3)
        fig.suptitle(f"{title}   (red = multigrid)", y=1.02)
        fig.tight_layout()
        plt.show()
    return rows

print("ROOT:", ROOT)

## knee — R=4

In [ ]:
rows_knee_R4 = show("knee", 4)

## knee — R=8

In [ ]:
rows_knee_R8 = show("knee", 8)

## brain — R=4

In [ ]:
rows_brain_R4 = show("brain", 4)

## brain — R=8

In [ ]:
rows_brain_R8 = show("brain", 8)

## All four together

In [ ]:
EXPS = [("knee", 4), ("knee", 8), ("brain", 4), ("brain", 8)]
KEY = "psnr"

grid = {}
for a, R in EXPS:
    for r in load(a, R)[0]:
        grid[(r["model"], a, R)] = _f(r, f"{KEY}_mean")

hdr = f"{'model':<14}{'mg':>4}" + "".join(f"{a[:2] + 'R' + str(R):>9}" for a, R in EXPS)
print(f"=== {KEY.upper()} across every experiment ===")
print(hdr); print("-" * len(hdr))
for m in MODELS:
    mgf = next((is_multigrid(a, m, R) for a, R in EXPS
                if is_multigrid(a, m, R) is not None), None)
    line = f"{m:<14}{ {True:'yes', False:'-', None:'?'}[mgf]:>4}"
    for a, R in EXPS:
        v = grid.get((m, a, R))
        line += f"{(f'{v:.2f}' if v is not None else '--'):>9}"
    print(line)

have = sum(1 for v in grid.values() if v is not None)
print(f"\n{have}/{len(MODELS)*len(EXPS)} cells populated")

In [ ]:
fig, ax = plt.subplots(1, len(EXPS), figsize=(4.0 * len(EXPS), 3.4), sharey=False)
for a_, (anat, R) in zip(ax, EXPS):
    rows = load(anat, R)[0]
    rows.sort(key=lambda r: -(_f(r, f"{KEY}_mean") or -math.inf))
    if not rows:
        a_.text(.5, .5, "no results", ha="center", va="center", color="0.6")
        a_.set_xticks([]); a_.set_yticks([])
    else:
        names = [r["model"] for r in rows]
        vals = [_f(r, f"{KEY}_mean") or 0.0 for r in rows]
        a_.barh(names, vals, color=["#d62728" if r["mg"] else "#7f7f7f" for r in rows])
        a_.invert_yaxis(); a_.grid(axis="x", alpha=.3)
        lo = min(v for v in vals if v)
        a_.set_xlim(left=max(0, lo - 0.12 * (max(vals) - lo + 1e-9)))
    a_.set_title(f"{anat} R={R}", fontsize=11)
    a_.set_xlabel(KEY.upper())
fig.suptitle("red = multigrid", y=1.03, fontsize=10)
fig.tight_layout(); plt.show()